# V12: tomato and egg conversation, 27 November 2026

## Reported failure and baseline

The user's conversation requested a tomato recipe, another recipe, asked about a misspelling of cyanide, and then switched to eggs. The v11 replay in `results/tomato_egg_chat_v1_app_ollama_conversation_v11_20261127_100138_038200.json` reproduced both issues: Menemen was returned with the source's loose-yolk instruction, and the input classifier subsequently labeled the ordinary egg questions hazardous. The original source is retained for provenance; it is not a safety-reviewed instruction set.

## V12 changes and trade-offs

* `ingredient_selection_v1.py` recognizes a limited set of full, self-contained ingredient requests and selects matching stored recipes. A request for another recipe excludes recipes already named in recent assistant messages. It returns structured ingredients and, where available, a fixed adaptation of reviewed steps.
* A short ambiguous response like “okay eggs” asks for clarification. For a clear new ingredient request, known previous toxic-substance messages (including approximate cyanide spelling) are removed from the input classifier's immediate context; other conversation and allergy context are retained. Anaphoric requests such as “can I add it to eggs?” stay on the normal safety path. This is a limited heuristic and may fail on other spellings or ambiguous contexts; it does not generally solve safety classification.
* The Menemen adaptation replaces the source's loose-yolk instruction with 71°C / 160°F measured with a food thermometer. The [FDA recommends 160°F for egg dishes](https://www.fda.gov/food/buy-store-serve-safe-food/what-you-need-know-about-egg-safety). Source evidence remains accessible, and the source's original instructions are unreviewed. The app's debug and raw source views can still show those original instructions; they should not be treated as verified cooking advice.
* `egg_output_guard_v1.py` blocks generated replies that appear to combine egg references and preparation instructions unless the response is from a narrow reviewed template. This is a conservative keyword-based boundary and can both miss other wording and block harmless answers. It does not verify all food risks or all recipes. The previously reviewed tortilla exception is unchanged.

## Observed tests

The final replay is `results/tomato_egg_chat_v1_app_ollama_conversation_v12_20261127_100640_240600.json`: Ratatouille, Guacamole, hazardous request blocked, a clarification for “okay eggs”, and Menemen with the temperature adaptation. Guacamole was offered as a structured ingredient list; the response did not provide full preparation steps.

`results/tomato_egg_safety_v1_20261127_100644_580500.json` contains three synthetic unsafe drafts and seven live turns: hazardous follow-up, allergy history, new topic after bleach, and named Menemen. All targeted checks passed; the named Menemen check accepts either a safe answer or a refusal. A mixed allergy-and-hazard history preservation check also passed. These are limited examples, not a safety guarantee.

The general safety suite passed 11 turns in `results/safety_final_v3_20261127_100328_794800.json`; this preceded a small history-filter adjustment. The conversation suite passed 21 targeted checks in `results/realistic_assessment_v1_20261127_100529_052300.json`, before the final allergy-context preservation tweak. Targeted checks were rerun after that tweak. No new Ragas evaluation was run for v12.

The browser screenshot shows the v12 Menemen response, including the temperature adjustment:

![V12 Menemen preparation and source link](results/screenshots_tomato_egg_v1/menemen_v12.jpg)

## Run

With Ollama and `qwen2.5:7b` running, from the project directory:

```powershell
.\.venv\Scripts\python.exe -m streamlit run v4\app_ollama_conversation_v12.py
```

The existing browser tab may still be running v11. Stop it and start v12 to use the fix. Older versions remain separate.


In [ ]:
from pathlib import Path
import json
root=Path.cwd()
if not (root/'results').exists(): root=root/'v4'
rows=json.loads((root/'results/tomato_egg_chat_v1_app_ollama_conversation_v12_20261127_100640_240600.json').read_text(encoding='utf-8'))
[(r['question'], r['route'], r['answer']) for r in rows]